In [0]:
%sql
select current_catalog() as cc,current_schema() as cs;


cc,cs
workspace,default


In [0]:
%sql
create schema if not exists workspace.job_skills;
create volume if not exists workspace.job_skills.raw_files;


In [0]:
%sql
use schema job_skills;



In [0]:
%sql
select current_catalog() as cc, current_schema() as cs;

cc,cs
workspace,job_skills


In [0]:
MY_ID="3026"
VOL="/Volumes/workspace/job_skills/raw_files"

In [0]:
from pyspark.sql import functions as F

VOL = "/Volumes/workspace/job_skills/raw_files"

# =========================================================
# 1. CONSTANTS
# =========================================================

FL = [
    "Python", "SQL", "Spark", "Airflow",
    "Kafka", "Hive", "AWS", "Azure",
    "GCP", "Docker", "Kubernetes", "Terraform",
    "Scala", "Java", "PowerBI", "Tableau"
]

SEC = [
    "BFSI", "IT Services", "Product", "Retail",
    "Health", "Telecom", "Energy", "Logistics"
]

CTY = [
    "Bengaluru", "Hyderabad", "Pune", "Chennai",
    "Mumbai", "Delhi NCR", "Noida", "Kolkata",
    "Ahmedabad", "Kochi"
]

# =========================================================
# 2. WRITE FUNCTION
# =========================================================

def wr(df, path):
    (
        df.write
        .mode("overwrite")
        .option("header", True)
        .option("ignoreLeadingWhiteSpace", False)
        .option("ignoreTrailingWhiteSpace", False)
        .csv(f"{VOL}/raw/{path}")
    )

# =========================================================
# 3. COMPANIES — 400
# =========================================================

companies = (
    spark.range(400)
    .selectExpr(
        "format_string('CMP%03d', id) AS company_id",
        "format_string('Company %03d', id + 1) AS company_name",
        "element_at(array("
        + ",".join([f"'{x}'" for x in SEC])
        + "), cast(id div 50 as int) + 1) AS sector",
        "element_at(array("
        + ",".join([f"'{x}'" for x in CTY])
        + "), cast(pmod(id, 10) as int) + 1) AS city"
    )
)

wr(companies, "companies")

print("Companies:", companies.count())
# =========================================================
# 4. BASE POSTINGS — 54,000
#    18 months × 3,000 postings
# =========================================================

base = (
    spark.range(54000)
    .selectExpr(
        "id",
        "cast(id div 3000 as int) AS m",
        "cast(pmod(id, 3000) as int) AS k"
    )
)

# Deterministic pseudo-random number.
# IMPORTANT: filler skills depend on ak, not k.
base = base.withColumn(
    "ak",
    F.pmod(
        (F.col("m") * 2700 + F.col("k")) * 7919,
        F.lit(48600)
    )
)

# =========================================================
# 5. TRACKED SKILL
# =========================================================

# One tracked skill based on the required k ranges.
#
# Snowflake:
#   k < 520 + 22*m
#
# Databricks:
#   1000 <= k < 1400 + 18*m
#
# Hadoop:
#   1800 <= k < 2660 - 38*m
#
# Otherwise choose another tracked skill deterministically.

tracked = (
    F.when(
        F.col("k") < 520 + 22 * F.col("m"),
        F.lit("Snowflake")
    )
    .when(
        (F.col("k") >= 1000) &
        (F.col("k") < 1400 + 18 * F.col("m")),
        F.lit("Databricks")
    )
    .when(
        (F.col("k") >= 1800) &
        (F.col("k") < 2660 - 38 * F.col("m")),
        F.lit("Hadoop")
    )
    .otherwise(
        F.element_at(
            F.array(*[F.lit(x) for x in FL]),
            (F.pmod(F.col("ak"), F.lit(16)) + 1).cast("int")
        )
    )
)

base = base.withColumn("tracked_skill", tracked)
# =========================================================
# 6. FILLER SKILLS — INDEPENDENT OF TRACKED-SKILL k RANGES
# =========================================================

skill_values = F.array(*[F.lit(x) for x in FL])

# Use a hash of posting id + seed.
# This is deliberately independent of k-based tracked-skill ranges.
filler_hash = F.xxhash64(
    F.col("id"),
    F.lit(26),
    F.lit("filler")
)

# Starting position among the 16 skills
start = F.pmod(filler_hash, F.lit(16))

# Generate 8 consecutive candidate positions.
# Since FL has 16 unique skills, these are unique.
candidate_array = F.array(
    F.element_at(skill_values, (F.pmod(start + 0, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 1, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 2, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 3, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 4, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 5, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 6, 16) + 1).cast("int")),
    F.element_at(skill_values, (F.pmod(start + 7, 16) + 1).cast("int"))
)

# Never allow the tracked skill to also appear as a filler.
candidate_array = F.array_except(
    candidate_array,
    F.array(F.col("tracked_skill"))
)

# Exactly 2, 3, or 4 fillers.
filler_count = F.pmod(
    F.xxhash64(
        F.col("id"),
        F.lit(26),
        F.lit("count")
    ),
    F.lit(3)
) + 2

base = base.withColumn(
    "filler_array",
    F.slice(
        candidate_array,
        1,
        filler_count
    )
)
# =========================================================
# 7. FINAL SKILL ARRAY
# =========================================================

# dbt is intentionally associated with Snowflake
# for the specified companion analysis.

base = base.withColumn(
    "skill_array",
    F.array_distinct(
        F.concat(
            F.col("filler_array"),
            F.array(F.col("tracked_skill")),
            F.when(
                (F.col("tracked_skill") == "Snowflake") &
                (F.pmod(F.col("k"), 2) == 0),
                F.array(F.lit("dbt"))
            ).otherwise(
                F.array()
            )
        )
    )

)
# =========================================================
# 8. POSTING COLUMNS
# =========================================================

base = base.withColumn(
    "posting_id",
    F.format_string("JP%06d", F.col("id"))
)

base = base.withColumn(
    "company_id",
    F.when(
        (F.col("k") >= 2775) &
        (F.col("k") < 2805),
        F.lit("CMP999")
    ).otherwise(
        F.format_string(
            "CMP%03d",
            F.pmod(F.col("id"), 400)
        )
    )
)

base = base.withColumn(
    "posted_date",
    F.when(
        (F.col("k") >= 2805) &
        (F.col("k") < 2820),
        F.lit("0000-00-00")
    ).otherwise(
        F.date_format(
            F.date_add(
                F.add_months(
                    F.to_date(F.lit("2024-01-01")),
                    F.col("m")
                ),
                F.pmod(F.col("id"), 28).cast("int")
            ),
            "yyyy-MM-dd"
        )
    )
)
# =========================================================
# 9. INTENTIONAL SKILLS CORRUPTION
# =========================================================

skills_joined = F.array_join(
    F.col("skill_array"),
    ";"
)

skills_comma = F.array_join(
    F.col("skill_array"),
    ","
)

skills_padded = F.concat(
    F.lit(" "),
    F.array_join(F.col("skill_array"), " ; "),
    F.lit(" ")
)

base = base.withColumn(
    "skills_raw",
    F.when(
        (F.col("k") >= 2700) &
        (F.col("k") < 2750),
        F.lit("")
    )
    .when(
        (F.col("k") >= 2750) &
        (F.col("k") < 2775),
        F.lit("Not specified")
    )
    .when(
        (F.col("k") < 2700) &
        (F.col("ak") < 1000),
        skills_comma
    )
    .when(
        (F.col("k") < 2700) &
        (F.col("ak") < 3200),
        F.upper(skills_joined)
    )
    .when(
        (F.col("k") < 2700) &
        (F.col("ak") < 4900),
        skills_padded
    )
    .otherwise(
        skills_joined
    )
)
base_postings = base.select(
    "posting_id",
    "company_id",
    "posted_date",
    "skills_raw"
)

print("Base postings:", base_postings.count())
# =========================================================
# 10. DUPLICATES — 1,350
# =========================================================

duplicates = (
    base
    .filter(
        (F.col("k") >= 2820) &
        (F.col("k") < 2895)
    )
    .select(
        "posting_id",
        "company_id",
        "posted_date",
        "skills_raw"
    )
)

print("Duplicates:", duplicates.count())
# =========================================================
# 11. FINAL RAW DATA
# =========================================================

final_postings = base_postings.unionByName(duplicates)

print("Final raw postings:", final_postings.count())

wr(final_postings, "postings")

Companies: 400
Base postings: 54000
Duplicates: 1350
Final raw postings: 55350


In [0]:
display(
    base
    .select(F.size("skill_array").alias("skill_count"))
    .groupBy("skill_count")
    .count()
    .orderBy("skill_count")
)

skill_count,count
3,15931
4,17878
5,18035
6,2156


In [0]:
display(
    base
    .filter(F.col("tracked_skill") == "Snowflake")
    .select(
        "posting_id",
        "tracked_skill",
        "skill_array"
    )
    .limit(10)
)

posting_id,tracked_skill,skill_array
JP000000,Snowflake,"List(Scala, Java, PowerBI, Tableau, Snowflake, dbt)"
JP000001,Snowflake,"List(Kafka, Hive, AWS, Snowflake)"
JP000002,Snowflake,"List(PowerBI, Tableau, Python, SQL, Snowflake, dbt)"
JP000003,Snowflake,"List(Kafka, Hive, AWS, Snowflake)"
JP000004,Snowflake,"List(Kubernetes, Terraform, Snowflake, dbt)"
JP000005,Snowflake,"List(Azure, GCP, Docker, Kubernetes, Snowflake)"
JP000006,Snowflake,"List(AWS, Azure, GCP, Snowflake, dbt)"
JP000007,Snowflake,"List(PowerBI, Tableau, Snowflake)"
JP000008,Snowflake,"List(GCP, Docker, Kubernetes, Snowflake, dbt)"
JP000009,Snowflake,"List(Kafka, Hive, AWS, Snowflake)"


In [0]:
display(
    base
    .filter(F.col("posting_id") == "JP036000")
    .select(
        "posting_id",
        "tracked_skill",
        "skill_array"
    )
)

posting_id,tracked_skill,skill_array
JP036000,Snowflake,"List(Kafka, Hive, Snowflake, dbt)"


In [0]:
display(
    base
    .filter(F.col("tracked_skill") == "Snowflake")
    .withColumn("has_dbt", F.array_contains("skill_array", "dbt"))
    .groupBy("has_dbt")
    .count()
)

has_dbt,count
true,6363
false,6363


In [0]:
display(
    base.filter(F.col("tracked_skill") == "Snowflake")
        .withColumn(
            "has_dbt",
            F.array_contains(F.col("skill_array"), "dbt")
        )
        
        .groupBy("has_dbt")
        .count()
)

has_dbt,count
true,6363
false,6363


In [0]:
display(
    base.filter(F.col("tracked_skill") == "Snowflake")
        .select("posting_id", "skill_array")
        .limit(20)
)

posting_id,skill_array
JP000000,"List(Python, SQL, Snowflake, dbt)"
JP000001,"List(Tableau, Python, SQL, Spark, Snowflake)"
JP000002,"List(PowerBI, Tableau, Python, Snowflake, dbt)"
JP000003,"List(Java, PowerBI, Snowflake)"
JP000004,"List(Scala, Java, PowerBI, Tableau, Snowflake, dbt)"
JP000005,"List(Terraform, Scala, Java, Snowflake)"
JP000006,"List(Kubernetes, Terraform, Snowflake, dbt)"
JP000007,"List(SQL, Spark, Airflow, Kafka, Snowflake)"
JP000008,"List(Python, SQL, Spark, Snowflake, dbt)"
JP000009,"List(Tableau, Python, Snowflake)"


In [0]:
raw = spark.read.option("header", True).csv(
    f"{VOL}/raw/postings"
)

print("Total:", raw.count())
print("Unique posting IDs:", raw.select("posting_id").distinct().count())
print(
    "Empty/null skills:",
    raw.filter(
        F.col("skills_raw").isNull() |
        (F.trim(F.col("skills_raw")) == "")
    ).count()
)

print("Not specified:",
      raw.filter(F.lower(F.trim(F.col("skills_raw"))) == "not specified").count())

print("CMP999:",
      raw.filter(F.col("company_id") == "CMP999").count())

print("Bad dates:",
      raw.filter(F.col("posted_date") == "0000-00-00").count())

Total: 55350
Unique posting IDs: 54000
Empty/null skills: 900
Not specified: 450
CMP999: 540
Bad dates: 270
